# SEC EDGAR Income Statement Extractor

This notebook downloads and parses income statements from SEC filings (10-K/10-Q) using edgartools.

## Features
- Handles both annual (10-K) and quarterly (10-Q) filings
- Extracts comprehensive income statement line items
- Works for any fiscal year or quarter
- Proper SEC identity setup to avoid 403 blocks

## 1. Setup and Imports

In [85]:
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
from edgar import Company, set_identity
from typing import Optional, Literal

# Load environment variables
load_dotenv()

# Set SEC identity to avoid 403 blocks
sec_identity = os.getenv("SEC_ID")
if not sec_identity:
    raise ValueError("SEC_ID not found in .env file")

set_identity(sec_identity)
print(f"✓ SEC identity set: {sec_identity}")

2026-01-29 05:40:41,910 - INFO - Identity of the Edgar REST client set to [pedroemail@duck.com]


✓ SEC identity set: pedroemail@duck.com


## 2. Define Income Statement Line Item Mapping

Map common XBRL tags to standardized income statement line items.

In [86]:
# XBRL concept mapping for income statement line items
INCOME_STATEMENT_MAPPING = {
    # Revenue Section
    "Revenue": [
        "Revenues",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "RevenueFromContractWithCustomerIncludingAssessedTax",
        "SalesRevenueNet",
        "SalesRevenueGoodsNet",
        "SalesRevenueServicesNet",
    ],
    "Cost of Revenue": [
        "CostOfRevenue",
        "CostOfGoodsAndServicesSold",
        "CostOfGoodsSold",
        "CostOfServices",
    ],
    "Gross Profit": [
        "GrossProfit",
    ],
    
    # Operating Expenses
    "Research & Development": [
        "ResearchAndDevelopmentExpense",
        "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    ],
    "Selling, General & Administrative": [
        "SellingGeneralAndAdministrativeExpense",
        "SellingAndMarketingExpense",
        "GeneralAndAdministrativeExpense",
    ],
    "Depreciation & Amortization": [
        "DepreciationDepletionAndAmortization",
        "DepreciationAndAmortization",
        "Depreciation",
        "AmortizationOfIntangibleAssets",
    ],
    "Other Operating Expenses": [
        "OtherOperatingIncomeExpenseNet",
        "OtherCostAndExpenseOperating",
    ],
    "Operating Income": [
        "OperatingIncomeLoss",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments",
    ],
    
    # Non-Operating Items
    "Interest Income": [
        "InterestIncomeOther",
        "InvestmentIncomeInterest",
        "InterestAndDividendIncomeOperating",
    ],
    "Interest Expense": [
        "InterestExpense",
        "InterestExpenseDebt",
    ],
    "Other Income/Expense": [
        "OtherNonoperatingIncomeExpense",
        "NonoperatingIncomeExpense",
        "OtherIncome",
    ],
    "Gains/Losses on Investments": [
        "GainLossOnInvestments",
        "GainLossOnSaleOfInvestments",
        "MarketableSecuritiesGainLoss",
    ],
    "Pretax Income": [
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments",
    ],
    
    # Taxes & Net Income
    "Income Tax Expense": [
        "IncomeTaxExpenseBenefit",
        "IncomeTaxesPaid",
    ],
    "Net Income from Continuing Operations": [
        "IncomeLossFromContinuingOperations",
        "IncomeLossFromContinuingOperationsIncludingPortionAttributableToNoncontrollingInterest",
    ],
    "Discontinued Operations": [
        "IncomeLossFromDiscontinuedOperationsNetOfTax",
        "DiscontinuedOperationIncomeLossFromDiscontinuedOperationDuringPhaseOutPeriodNetOfTax",
    ],
    "Net Income": [
        "NetIncomeLoss",
        "ProfitLoss",
        "NetIncomeLossAvailableToCommonStockholdersBasic",
    ],
    "Net Income Attributable to Noncontrolling Interests": [
        "NetIncomeLossAttributableToNoncontrollingInterest",
        "MinorityInterestNetOfTax",
    ],
    "Net Income Attributable to Common Shareholders": [
        "NetIncomeLossAvailableToCommonStockholdersBasic",
        "NetIncomeLossAvailableToCommonStockholdersDiluted",
    ],
    
    # Per Share Data
    "Basic EPS": [
        "EarningsPerShareBasic",
    ],
    "Diluted EPS": [
        "EarningsPerShareDiluted",
    ],
    "Basic Weighted Average Shares Outstanding": [
        "WeightedAverageNumberOfSharesOutstandingBasic",
        "WeightedAverageNumberOfShareOutstandingBasicAndDiluted",
    ],
    "Diluted Weighted Average Shares Outstanding": [
        "WeightedAverageNumberOfDilutedSharesOutstanding",
    ],
}

print(f"✓ Defined {len(INCOME_STATEMENT_MAPPING)} income statement line items")

✓ Defined 23 income statement line items


## 3. Function to Get Filing

In [108]:
def parse_date(date_str):
    """Parse a date string to datetime object."""
    if isinstance(date_str, str):
        try:
            return pd.to_datetime(date_str)
        except Exception:
            return None
    return date_str

def get_filing(
    ticker: str,
    filing_type: Literal["10-K", "10-Q"],
    year: Optional[int] = None,
    quarter: Optional[int] = None
):
    """
    Retrieve a specific SEC filing for a company.
    
    Args:
        ticker: Company ticker symbol
        filing_type: Either "10-K" (annual) or "10-Q" (quarterly)
        year: Fiscal year (if None, gets the most recent)
        quarter: Fiscal quarter (1-4, only for 10-Q)
        
    Returns:
        Filing object from edgartools
        
    Raises:
        ValueError: If parameters are invalid
        FileNotFoundError: If filing is not found
    """
    if filing_type == "10-Q" and quarter is not None:
        if not 1 <= quarter <= 4:
            raise ValueError("Quarter must be between 1 and 4")
    
    # Get company
    company = Company(ticker)
    print(f"✓ Retrieved company: {company.name} ({ticker})")
    
    # Get filings of the specified type
    filings = company.get_filings(form=filing_type)
    
    if filings.empty:
        raise FileNotFoundError(f"No {filing_type} filings found for {ticker}")
    
    # Filter by year if specified
    if year is not None:
        filings_filtered = []
        
        for f in filings:
            # Parse dates
            filing_date = parse_date(f.filing_date) if hasattr(f, 'filing_date') else None
            period_date = parse_date(f.period_of_report) if hasattr(f, 'period_of_report') else None
            
            # Check if either date matches the year
            filing_year_match = filing_date and filing_date.year == year
            period_year_match = period_date and period_date.year == year
            
            if filing_year_match or period_year_match:
                # Store the parsed dates for later use
                f._parsed_filing_date = filing_date
                f._parsed_period_date = period_date
                filings_filtered.append(f)
        
        if not filings_filtered:
            raise FileNotFoundError(f"No {filing_type} filings found for {ticker} in year {year}")
        
        # For 10-Q, further filter by quarter if specified
        if filing_type == "10-Q" and quarter is not None:
            quarter_filtered = []
            
            for f in filings_filtered:
                period_date = f._parsed_period_date
                if period_date:
                    # Calculate quarter from month (Q1: Jan-Mar, Q2: Apr-Jun, Q3: Jul-Sep, Q4: Oct-Dec)
                    file_quarter = ((period_date.month - 1) // 3) + 1
                    if file_quarter == quarter:
                        quarter_filtered.append(f)
            
            if not quarter_filtered:
                raise FileNotFoundError(
                    f"No {filing_type} filings found for {ticker} in Q{quarter} {year}"
                )
            
            filings_filtered = quarter_filtered
        
        filing = filings_filtered[0]
    else:
        # Get the most recent filing
        filing = filings[0]
    
    filing_date = filing.filing_date if hasattr(filing, 'filing_date') else 'Unknown'
    period = filing.period_of_report if hasattr(filing, 'period_of_report') else 'Unknown'
    
    print(f"✓ Retrieved {filing_type} filing")
    print(f"  Filing Date: {filing_date}")
    print(f"  Period of Report: {period}")
    
    return filing

# Test the function with a well-known company
test_filing = get_filing("AAPL", "10-K", year=2023)
print(f"\n✓ Test successful")

2026-01-29 06:14:50,619 - INFO - No cache policy for data.sec.gov:///submissions/CIK0000320193.json, not retrieving from cache
2026-01-29 06:14:50,620 - INFO - Making HTTP Request <Request('GET', 'https://data.sec.gov/submissions/CIK0000320193.json')>
2026-01-29 06:14:50,833 - INFO - HTTP Request: GET https://data.sec.gov/submissions/CIK0000320193.json "HTTP/1.1 200 OK"
2026-01-29 06:14:50,858 - INFO - No cache policy for data.sec.gov:///submissions/CIK0000320193-submissions-001.json, not retrieving from cache
2026-01-29 06:14:50,858 - INFO - Making HTTP Request <Request('GET', 'https://data.sec.gov/submissions/CIK0000320193-submissions-001.json')>
2026-01-29 06:14:50,960 - INFO - HTTP Request: GET https://data.sec.gov/submissions/CIK0000320193-submissions-001.json "HTTP/1.1 200 OK"
2026-01-29 06:14:50,971 - INFO - matched .*www\.sec\.gov, using value www.sec.gov: {'/submissions.*': 30, '/include/ticker\\.txt.*': 30, '/files/company_tickers\\.json.*': 30, '.*index/.*': 1800, '/Archives

✓ Retrieved company: Apple Inc. (AAPL)


2026-01-29 06:14:51,059 - INFO - /Archives/edgar/data/320193/000032019324000123/0000320193-24-000123.txt matched /Archives/edgar/data, using value True
2026-01-29 06:14:51,061 - INFO - Cache policy allows unlimited cache, returning /home/pedro/.edgar/_tcache/www.sec.gov/Archives-edgar-data-320193-000032019324000123-0000320193-24-000123.txt
2026-01-29 06:14:51,077 - INFO - HTTP Request: GET https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/0000320193-24-000123.txt "HTTP/1.1 200 OK"
2026-01-29 06:14:51,131 - INFO - matched .*www\.sec\.gov, using value www.sec.gov: {'/submissions.*': 30, '/include/ticker\\.txt.*': 30, '/files/company_tickers\\.json.*': 30, '.*index/.*': 1800, '/Archives/edgar/data': True}
2026-01-29 06:14:51,132 - INFO - /Archives/edgar/data/320193/000032019323000106/0000320193-23-000106.txt matched /Archives/edgar/data, using value True
2026-01-29 06:14:51,134 - INFO - Cache policy allows unlimited cache, returning /home/pedro/.edgar/_tcache/www.sec.gov/A

✓ Retrieved 10-K filing
  Filing Date: 2023-11-03
  Period of Report: 2023-09-30

✓ Test successful


## 4. Extract Income Statement Data

In [109]:
def get_fact_value(xbrl, concept_name: str, end_date: str, prefer_shortest: bool = True):
    """
    Get a specific fact value from XBRL, preferring the shortest duration for the given end date.
    
    Args:
        xbrl: XBRL object
        concept_name: The XBRL concept to search for  
        end_date: The end date in YYYY-MM-DD format
        prefer_shortest: If True, prefer the shortest duration period
        
    Returns:
        The value, or None if not found
    """
    try:
        # Query facts for the concept
        facts = xbrl.facts.query().by_concept(concept_name).to_dataframe()
        
        if facts.empty:
            return None
        
        # Filter by end date
        date_filtered = facts[facts['end'] == end_date] if 'end' in facts.columns else facts
        
        if date_filtered.empty:
            return None
        
        # Filter out dimensional breakdowns (segments, product lines, etc.)
        # We only want consolidated totals
        if 'dimensions' in date_filtered.columns:
            # Prefer facts with no dimensions (consolidated totals)
            no_dimensions = date_filtered[date_filtered['dimensions'].isna() | (date_filtered['dimensions'] == '{}') | (date_filtered['dimensions'] == '')]
            if not no_dimensions.empty:
                date_filtered = no_dimensions
        
        # If prefer_shortest, sort by duration and take the shortest
        if prefer_shortest and 'duration' in date_filtered.columns:
            # Sort by duration (shortest first)
            date_filtered = date_filtered.sort_values('duration')
        
        # Get the first (shortest duration, non-dimensional) value
        if 'value' in date_filtered.columns:
            return date_filtered.iloc[0]['value']
        
        return None
        
    except Exception:
        return None

def extract_income_statement(filing) -> pd.DataFrame:
    """
    Extract income statement data from a SEC filing.
    
    Args:
        filing: Filing object from edgartools
        
    Returns:
        DataFrame with income statement line items and values
        
    Raises:
        ValueError: If financials cannot be extracted
    """
    # Get the XBRL data
    try:
        xbrl = filing.xbrl()
        if xbrl is None:
            raise ValueError("Unable to extract XBRL data from filing")
    except Exception as e:
        raise ValueError(f"Error accessing XBRL data: {e}")
    
    # Get the income statement using edgartools API
    try:
        # Get the income statement - use the most appropriate period
        # For 10-Q, get period views to access the quarterly (not YTD) data
        try:
            period_views = xbrl.get_period_views('income')
            
            # For 10-Q, there will be multiple periods (quarterly and YTD)
            # We want the shortest duration period (quarterly, not YTD)
            if period_views and len(period_views) > 0:
                # The first period view is typically the current/shortest period
                current_view = period_views[0]
                income_stmt = current_view.income_statement()
            else:
                # Fallback to standard method
                income_stmt = xbrl.statements.income_statement()
        except (TypeError, AttributeError):
            # If get_period_views doesn't work, use standard method
            income_stmt = xbrl.statements.income_statement()
        
        # Method 2: If that fails or returns empty, try finding by statement name
        if income_stmt is None:
            # Try to find condensed consolidated statements (common in 10-Q)
            for stmt_name in ['CONDENSEDCONSOLIDATEDSTATEMENTSOFOPERATIONSUnaudited',
                             'CONSOLIDATEDSTATEMENTSOFINCOME',
                             'CONSOLIDATEDSTATEMENTSOFOPERATIONS',
                             'StatementsOfIncome',
                             'StatementsOfOperations']:
                try:
                    income_stmt = xbrl.get_statement(stmt_name)
                    if income_stmt is not None:
                        break
                except (AttributeError, KeyError):
                    continue
        
        if income_stmt is None:
            raise ValueError("Income statement is not available")
        
        # Convert to DataFrame
        income_df = income_stmt.to_dataframe()
        
        if income_df.empty:
            raise ValueError("Income statement DataFrame is empty")
            
    except Exception as e:
        raise ValueError(f"Error retrieving income statement: {e}")
    
    print(f"✓ Retrieved income statement")
    print(f"  Shape: {income_df.shape}")
    
    # Identify date columns (columns that look like dates: YYYY-MM-DD format)
    date_columns = [
        col for col in income_df.columns 
        if isinstance(col, str) and len(col) == 10 and col[4] == '-' and col[7] == '-'
    ]
    
    if not date_columns:
        raise ValueError("No date columns found in income statement")
    
    # Sort date columns to get the most recent
    date_columns_sorted = sorted(date_columns, reverse=True)
    most_recent_period = date_columns_sorted[0]
    
    print(f"  Date columns found: {date_columns_sorted}")
    print(f"  Using most recent period: {most_recent_period}")
    print(f"  Using facts API to extract values with shortest duration...")
    
    # Extract line items using facts API (ensures shortest duration for 10-Q)
    results = []
    
    for line_item, concept_list in INCOME_STATEMENT_MAPPING.items():
        value = None
        matched_concept = None
        
        # Try each concept in the list
        for concept in concept_list:
            # Build the full concept name (us-gaap prefix)
            full_concept = f"us-gaap_{concept}"
            
            # Query using facts API with shortest duration preference
            fact_value = get_fact_value(xbrl, full_concept, most_recent_period, prefer_shortest=True)
            
            if fact_value is not None:
                matched_concept = full_concept
                value = fact_value
                break
        
        results.append({
            'Line Item': line_item,
            'Value': value,
            'XBRL Concept': matched_concept,
        })
    
    # Convert to DataFrame
    result_df = (
        pd.DataFrame(results)
        .pipe(lambda df: df.assign(
            Value=pd.to_numeric(df['Value'], errors='coerce')
        ))
    )
    
    found_count = result_df['Value'].notna().sum()
    total_count = len(result_df)
    
    print(f"✓ Extracted {found_count}/{total_count} line items")
    
    return result_df

# Test with the previously retrieved filing
test_income_stmt = extract_income_statement(test_filing)
print(f"\n✓ Test successful")
test_income_stmt.head(10)

✓ Retrieved income statement
  Shape: (42, 19)
  Date columns found: ['2023-09-30', '2022-09-24', '2021-09-25']
  Using most recent period: 2023-09-30
  Using facts API to extract values with shortest duration...
✓ Extracted 18/23 line items

✓ Test successful


,Line Item,Value,XBRL Concept
0,Revenue,2.980850e+11,us-gaap_RevenueFromContractWithCustomerExcludi...
1,Cost of Revenue,1.892820e+11,us-gaap_CostOfGoodsAndServicesSold
2,Gross Profit,1.691480e+11,us-gaap_GrossProfit
3,Research & Development,2.991500e+10,us-gaap_ResearchAndDevelopmentExpense
4,"Selling, General & Administrative",2.493200e+10,us-gaap_SellingGeneralAndAdministrativeExpense
5,Depreciation & Amortization,1.151900e+10,us-gaap_DepreciationDepletionAndAmortization
6,Other Operating Expenses,NaN,NaN
7,Operating Income,6.050800e+10,us-gaap_OperatingIncomeLoss
8,Interest Income,3.750000e+09,us-gaap_InvestmentIncomeInterest
9,Interest Expense,3.933000e+09,us-gaap_InterestExpense


## 4.5 Diagnostic - Explore XBRL Structure (Run if errors occur)

Use this cell to understand the structure of the XBRL data.

In [104]:
# Diagnostic: Explore the XBRL object structure
print("=== XBRL Object Exploration ===\n")

xbrl = test_filing.xbrl()
print(f"XBRL object type: {type(xbrl)}")
print(f"XBRL object: {xbrl}\n")

print("Available attributes and methods (non-private):")
attrs = [attr for attr in dir(xbrl) if not attr.startswith('_')]
for attr in attrs:
    attr_type = type(getattr(xbrl, attr, None)).__name__
    print(f"  - {attr:<30} ({attr_type})")

# Try common financial statement accessors
print("\n" + "="*50)
print("Trying common accessors:")
print("="*50)

# Try: statements
if hasattr(xbrl, 'statements'):
    print("\n✓ xbrl.statements found:")
    try:
        stmts = xbrl.statements
        print(f"  Type: {type(stmts)}")
        if hasattr(stmts, '__iter__'):
            for stmt in stmts:
                print(f"  - {stmt}")
    except Exception as e:
        print(f"  Error: {e}")

# Try: get_statement method
if hasattr(xbrl, 'get_statement'):
    print("\n✓ xbrl.get_statement() method found")

# Try: income_statement property
for prop_name in ['income_statement', 'income', 'IncomeStatement']:
    if hasattr(xbrl, prop_name):
        print(f"\n✓ xbrl.{prop_name} found:")
        try:
            stmt = getattr(xbrl, prop_name)
            print(f"  Type: {type(stmt)}")
            if isinstance(stmt, pd.DataFrame):
                print(f"  Shape: {stmt.shape}")
                print(f"  Columns: {list(stmt.columns)}")
                print(f"\n  First few rows:")
                print(stmt.head())
        except Exception as e:
            print(f"  Error: {e}")

# Try: facts
if hasattr(xbrl, 'facts'):
    print("\n✓ xbrl.facts found:")
    try:
        facts = xbrl.facts
        print(f"  Type: {type(facts)}")
        if isinstance(facts, pd.DataFrame):
            print(f"  Shape: {facts.shape}")
            print(f"  Columns: {list(facts.columns)}")
            # Show sample revenue-related facts
            if 'concept' in facts.columns:
                revenue_facts = facts[facts['concept'].str.contains('Revenue', case=False, na=False)]
                print(f"\n  Sample Revenue concepts found: {len(revenue_facts)}")
                if not revenue_facts.empty:
                    print(revenue_facts.head())
    except Exception as e:
        print(f"  Error: {e}")

print("\n" + "="*50)

=== XBRL Object Exploration ===

XBRL object type: <class 'edgar.xbrl.xbrl.XBRL'>


XBRL object: ╭───────────────────────────────────────────────── XBRL Document ─────────────────────────────────────────────────╮
│ Apple Inc. (AAPL) • CIK 0000320193                                                                              │
│                                                                                                                 │
│          Form:  10-K                                                                                            │
│ Fiscal Period:  Fiscal Year 2023 (ended Sep 30, 2023)                                                           │
│          Data:  1,164 facts • 205 contexts                                                                      │
│                                                                                                                 │
│ Periods Available for Statements:                                                                               │
│   Annual: FY 2023, FY 2022, FY 2021                      

## 4.6 Diagnostic - View Raw Income Statement Data

Use this cell to see the actual data structure and verify values.

In [105]:
# Diagnostic: View the raw income statement data
xbrl = filing.xbrl()
income_stmt = xbrl.statements.income_statement()
income_df = income_stmt.to_dataframe()

print("="*80)
print("ALL COLUMNS IN INCOME STATEMENT:")
print("="*80)
print(f"Shape: {income_df.shape}")
print(f"\nAll columns: {list(income_df.columns)}")

# Show all rows with a few key columns
print("\n" + "="*80)
print("INCOME STATEMENT DATA (showing first 50 rows):")
print("="*80)

# Select columns to display
display_cols = ['concept', 'label']
# Add all date-like columns
date_cols = [col for col in income_df.columns 
             if isinstance(col, str) and len(col) == 10 and col[4] == '-' and col[7] == '-']
display_cols.extend(date_cols)

print(f"\nDate columns found: {date_cols}")
print("\n")

# Display the data
income_df[display_cols].head(50)

ALL COLUMNS IN INCOME STATEMENT:
Shape: (39, 18)

All columns: ['concept', 'label', 'standard_concept', '2025-06-28', '2024-06-29', 'level', 'abstract', 'dimension', 'is_breakdown', 'dimension_axis', 'dimension_member', 'dimension_member_label', 'dimension_label', 'balance', 'weight', 'preferred_sign', 'parent_concept', 'parent_abstract_concept']

INCOME STATEMENT DATA (showing first 50 rows):

Date columns found: ['2025-06-28', '2024-06-29']




,concept,label,2025-06-28,2024-06-29
0,us-gaap_RevenueFromContractWithCustomerExcludi...,Net sales,3.136950e+11,2.961050e+11
1,us-gaap_RevenueFromContractWithCustomerExcludi...,Products,2.332870e+11,2.249080e+11
2,us-gaap_RevenueFromContractWithCustomerExcludi...,Services,8.040800e+10,7.119700e+10
3,us-gaap_RevenueFromContractWithCustomerExcludi...,iPhone,1.605610e+11,1.549610e+11
4,us-gaap_RevenueFromContractWithCustomerExcludi...,Mac,2.498200e+10,2.224000e+10
5,us-gaap_RevenueFromContractWithCustomerExcludi...,iPad,2.107100e+10,1.974400e+10
6,us-gaap_RevenueFromContractWithCustomerExcludi...,"Wearables, Home and Accessories",2.667300e+10,2.796300e+10
7,us-gaap_RevenueFromContractWithCustomerExcludi...,Americas,1.341610e+11,1.253810e+11
8,us-gaap_RevenueFromContractWithCustomerExcludi...,Europe,8.232900e+10,7.640400e+10
9,us-gaap_RevenueFromContractWithCustomerExcludi...,Greater China,4.988400e+10,5.191900e+10


## 5. User Interface - Get Income Statement

Configure the parameters below to retrieve the income statement for your desired company and period.

In [110]:
# ===== USER CONFIGURATION =====
# Modify these parameters to get the income statement you need

TICKER = "AAPL"           # Company ticker symbol
FILING_TYPE = "10-Q"      # "10-K" for annual, "10-Q" for quarterly
YEAR = 2025            # Fiscal year (set to None for most recent)
QUARTER = 2              # Fiscal quarter 1-4 (only for 10-Q, set to None for most recent)

# ===== END USER CONFIGURATION =====

print("Configuration:")
print(f"  Ticker: {TICKER}")
print(f"  Filing Type: {FILING_TYPE}")
print(f"  Year: {YEAR if YEAR else 'Most Recent'}")
if FILING_TYPE == "10-Q":
    print(f"  Quarter: Q{QUARTER if QUARTER else 'Most Recent'}")

Configuration:
  Ticker: AAPL
  Filing Type: 10-Q
  Year: 2025
  Quarter: Q2


In [111]:
# Retrieve the filing
filing = get_filing(
    ticker=TICKER,
    filing_type=FILING_TYPE,
    year=YEAR,
    quarter=QUARTER
)

# Extract the income statement
income_statement_df = extract_income_statement(filing)

# Display results
print("\n" + "="*80)
print("INCOME STATEMENT")
print("="*80)
income_statement_df

2026-01-29 06:15:46,152 - INFO - No cache policy for data.sec.gov:///submissions/CIK0000320193.json, not retrieving from cache
2026-01-29 06:15:46,153 - INFO - Making HTTP Request <Request('GET', 'https://data.sec.gov/submissions/CIK0000320193.json')>
2026-01-29 06:15:46,357 - INFO - HTTP Request: GET https://data.sec.gov/submissions/CIK0000320193.json "HTTP/1.1 200 OK"
2026-01-29 06:15:46,384 - INFO - No cache policy for data.sec.gov:///submissions/CIK0000320193-submissions-001.json, not retrieving from cache
2026-01-29 06:15:46,384 - INFO - Making HTTP Request <Request('GET', 'https://data.sec.gov/submissions/CIK0000320193-submissions-001.json')>
2026-01-29 06:15:46,528 - INFO - HTTP Request: GET https://data.sec.gov/submissions/CIK0000320193-submissions-001.json "HTTP/1.1 200 OK"
2026-01-29 06:15:46,537 - INFO - matched .*www\.sec\.gov, using value www.sec.gov: {'/submissions.*': 30, '/include/ticker\\.txt.*': 30, '/files/company_tickers\\.json.*': 30, '.*index/.*': 1800, '/Archives

✓ Retrieved company: Apple Inc. (AAPL)


2026-01-29 06:15:46,610 - INFO - matched .*www\.sec\.gov, using value www.sec.gov: {'/submissions.*': 30, '/include/ticker\\.txt.*': 30, '/files/company_tickers\\.json.*': 30, '.*index/.*': 1800, '/Archives/edgar/data': True}
2026-01-29 06:15:46,611 - INFO - /Archives/edgar/data/320193/000032019325000008/0000320193-25-000008.txt matched /Archives/edgar/data, using value True
2026-01-29 06:15:46,613 - INFO - Cache policy allows unlimited cache, returning /home/pedro/.edgar/_tcache/www.sec.gov/Archives-edgar-data-320193-000032019325000008-0000320193-25-000008.txt
2026-01-29 06:15:46,621 - INFO - HTTP Request: GET https://www.sec.gov/Archives/edgar/data/320193/000032019325000008/0000320193-25-000008.txt "HTTP/1.1 200 OK"
2026-01-29 06:15:46,669 - INFO - matched .*www\.sec\.gov, using value www.sec.gov: {'/submissions.*': 30, '/include/ticker\\.txt.*': 30, '/files/company_tickers\\.json.*': 30, '.*index/.*': 1800, '/Archives/edgar/data': True}
2026-01-29 06:15:46,670 - INFO - /Archives/edg

✓ Retrieved 10-Q filing
  Filing Date: 2025-08-01
  Period of Report: 2025-06-28
✓ Retrieved income statement
  Shape: (39, 18)
  Date columns found: ['2025-06-28', '2024-06-29']
  Using most recent period: 2025-06-28
  Using facts API to extract values with shortest duration...
✓ Extracted 16/23 line items

INCOME STATEMENT


,Line Item,Value,XBRL Concept
0,Revenue,6.661300e+10,us-gaap_RevenueFromContractWithCustomerExcludi...
1,Cost of Revenue,4.362000e+10,us-gaap_CostOfGoodsAndServicesSold
2,Gross Profit,4.371800e+10,us-gaap_GrossProfit
3,Research & Development,8.866000e+09,us-gaap_ResearchAndDevelopmentExpense
4,"Selling, General & Administrative",6.650000e+09,us-gaap_SellingGeneralAndAdministrativeExpense
5,Depreciation & Amortization,8.571000e+09,us-gaap_DepreciationDepletionAndAmortization
6,Other Operating Expenses,NaN,NaN
7,Operating Income,1.651100e+10,us-gaap_OperatingIncomeLoss
8,Interest Income,NaN,NaN
9,Interest Expense,NaN,NaN


## 6. Format and Display Results

Format the income statement for better readability with proper currency formatting.

In [98]:
def format_income_statement(df: pd.DataFrame) -> pd.DataFrame:
    """
    Format income statement for better display.
    
    Args:
        df: DataFrame with income statement data
        
    Returns:
        Formatted DataFrame with styled values
    """
    formatted_df = (
        df
        .copy()
        .assign(
            # Format values as currency (assuming values are in dollars)
            Formatted_Value=lambda x: x['Value'].apply(
                lambda v: f"${v:,.0f}" if pd.notna(v) else "Not Found"
            ),
            # Add status indicator
            Status=lambda x: x['Value'].apply(
                lambda v: "✓" if pd.notna(v) else "✗"
            )
        )
        # Select and reorder columns
        [['Status', 'Line Item', 'Formatted_Value', 'XBRL Concept']]
        .rename(columns={'Formatted_Value': 'Value'})
    )
    
    return formatted_df

# Format the income statement
formatted_income_statement = format_income_statement(income_statement_df)

# Display with better formatting
print(f"\n{'='*100}")
print(f"INCOME STATEMENT - {TICKER}")
print(f"Filing Type: {FILING_TYPE} | Year: {YEAR if YEAR else 'Most Recent'}")
print(f"{'='*100}\n")

formatted_income_statement


INCOME STATEMENT - AAPL
Filing Type: 10-Q | Year: 2025



,Status,Line Item,Value,XBRL Concept
0,✓,Revenue,"$313,695,000,000",us-gaap_RevenueFromContractWithCustomerExcludi...
1,✓,Cost of Revenue,"$166,835,000,000",us-gaap_CostOfGoodsAndServicesSold
2,✓,Gross Profit,"$146,860,000,000",us-gaap_GrossProfit
3,✓,Research & Development,"$25,684,000,000",us-gaap_ResearchAndDevelopmentExpense
4,✓,"Selling, General & Administrative","$20,553,000,000",us-gaap_SellingGeneralAndAdministrativeExpense
5,✗,Depreciation & Amortization,Not Found,NaN
6,✗,Other Operating Expenses,Not Found,NaN
7,✓,Operating Income,"$100,623,000,000",us-gaap_OperatingIncomeLoss
8,✗,Interest Income,Not Found,NaN
9,✗,Interest Expense,Not Found,NaN


## 7. Validation Summary

Check which required line items were found and which are missing.

In [50]:
def validate_income_statement(df: pd.DataFrame) -> dict:
    """
    Validate income statement completeness.
    
    Args:
        df: DataFrame with income statement data
        
    Returns:
        Dictionary with validation statistics
    """
    total_items = len(df)
    found_items = df['Value'].notna().sum()
    missing_items = total_items - found_items
    completeness_pct = (found_items / total_items) * 100 if total_items > 0 else 0
    
    missing_line_items = df[df['Value'].isna()]['Line Item'].tolist()
    
    return {
        'total_items': total_items,
        'found_items': found_items,
        'missing_items': missing_items,
        'completeness_percentage': completeness_pct,
        'missing_line_items': missing_line_items
    }

# Validate the income statement
validation_results = validate_income_statement(income_statement_df)

print("\n" + "="*80)
print("VALIDATION SUMMARY")
print("="*80)
print(f"Total Line Items:        {validation_results['total_items']}")
print(f"Found:                   {validation_results['found_items']} ✓")
print(f"Missing:                 {validation_results['missing_items']} ✗")
print(f"Completeness:            {validation_results['completeness_percentage']:.1f}%")

if validation_results['missing_line_items']:
    print("\nMissing Line Items:")
    for item in validation_results['missing_line_items']:
        print(f"  - {item}")
else:
    print("\n✓ All required line items found!")

print("="*80)


VALIDATION SUMMARY
Total Line Items:        23
Found:                   15 ✓
Missing:                 8 ✗
Completeness:            65.2%

Missing Line Items:
  - Depreciation & Amortization
  - Other Operating Expenses
  - Interest Income
  - Interest Expense
  - Gains/Losses on Investments
  - Discontinued Operations
  - Net Income Attributable to Noncontrolling Interests
  - Net Income Attributable to Common Shareholders


## 8. Export Data (Optional)

Export the income statement to CSV for further analysis.

In [ ]:
# Optional: Export to CSV
EXPORT_TO_CSV = False  # Set to True to enable export
OUTPUT_FILENAME = f"{TICKER}_{FILING_TYPE}_{YEAR}_income_statement.csv"

if EXPORT_TO_CSV:
    output_path = Path(OUTPUT_FILENAME)
    income_statement_df.to_csv(output_path, index=False)
    print(f"✓ Income statement exported to: {output_path.absolute()}")
else:
    print("Export disabled. Set EXPORT_TO_CSV = True to enable.")

## 9. Examples and Usage Patterns

Here are some common usage examples you can copy and modify.

In [ ]:
# EXAMPLE 1: Get most recent 10-K for Apple
# filing_ex1 = get_filing("AAPL", "10-K")
# income_ex1 = extract_income_statement(filing_ex1)

# EXAMPLE 2: Get specific year 10-K for Microsoft
# filing_ex2 = get_filing("MSFT", "10-K", year=2022)
# income_ex2 = extract_income_statement(filing_ex2)

# EXAMPLE 3: Get most recent 10-Q for Tesla
# filing_ex3 = get_filing("TSLA", "10-Q")
# income_ex3 = extract_income_statement(filing_ex3)

# EXAMPLE 4: Get specific quarter 10-Q for Amazon
# filing_ex4 = get_filing("AMZN", "10-Q", year=2023, quarter=2)
# income_ex4 = extract_income_statement(filing_ex4)

# EXAMPLE 5: Compare multiple quarters
# quarters_data = []
# for q in [1, 2, 3, 4]:
#     try:
#         filing = get_filing("GOOGL", "10-Q", year=2023, quarter=q)
#         income = extract_income_statement(filing)
#         income['Quarter'] = f"Q{q}"
#         quarters_data.append(income)
#     except FileNotFoundError:
#         print(f"Q{q} not found")
# 
# if quarters_data:
#     combined_quarters = pd.concat(quarters_data, ignore_index=True)
#     print(combined_quarters)

print("✓ Example patterns defined (uncomment to run)")

## 10. Troubleshooting and Diagnostics

Use these cells to diagnose issues with data extraction.

In [ ]:
# Diagnostic: View all available concepts in the income statement
def show_all_concepts(filing):
    """Display all XBRL concepts available in the income statement."""
    xbrl = filing.xbrl()
    income_stmt = xbrl.get_income_statement()
    
    if income_stmt is not None and not income_stmt.empty:
        concepts = income_stmt['concept'].unique() if 'concept' in income_stmt.columns else []
        print(f"Total unique concepts found: {len(concepts)}\n")
        
        for i, concept in enumerate(sorted(concepts), 1):
            print(f"{i:3d}. {concept}")
        
        return income_stmt
    else:
        print("No income statement data available")
        return None

# Uncomment to run diagnostics on the current filing:
# all_concepts_df = show_all_concepts(filing)
# all_concepts_df.head(20)

print("✓ Diagnostic functions ready (uncomment to run)")

In [ ]:
# Diagnostic: Search for specific concepts by keyword
def search_concepts(filing, keyword: str):
    """
    Search for XBRL concepts containing a specific keyword.
    
    Args:
        filing: Filing object
        keyword: Search term (case-insensitive)
    """
    xbrl = filing.xbrl()
    income_stmt = xbrl.get_income_statement()
    
    if income_stmt is not None and not income_stmt.empty and 'concept' in income_stmt.columns:
        matches = income_stmt[
            income_stmt['concept'].str.contains(keyword, case=False, na=False)
        ]
        
        if not matches.empty:
            print(f"Found {len(matches)} matches for '{keyword}':\n")
            return matches[['concept', 'value', 'label']].drop_duplicates(subset=['concept'])
        else:
            print(f"No matches found for '{keyword}'")
            return pd.DataFrame()
    else:
        print("No income statement data available")
        return pd.DataFrame()

# Example: Uncomment to search for specific concepts
# search_concepts(filing, "Revenue")
# search_concepts(filing, "EarningsPerShare")
# search_concepts(filing, "Interest")

print("✓ Search function ready (uncomment examples to run)")

## Summary

This notebook provides a complete solution for extracting income statements from SEC filings using edgartools.

### Key Features:
- ✓ Handles both 10-K (annual) and 10-Q (quarterly) filings
- ✓ Works for any fiscal year or quarter
- ✓ Extracts all required income statement line items
- ✓ Proper SEC identity setup (no 403 errors)
- ✓ Comprehensive error handling
- ✓ Validation and completeness checking
- ✓ Diagnostic tools for troubleshooting
- ✓ Export functionality

### How to Use:
1. Run cells 1-8 to set up functions (one time)
2. Modify the parameters in cell 10 (TICKER, FILING_TYPE, YEAR, QUARTER)
3. Run cells 11-15 to retrieve and display the income statement
4. Optional: Enable CSV export in cell 17

### Notes:
- Some line items may not be available for all companies (accounting standards vary)
- Use the diagnostic tools in cells 21-22 to explore what concepts are available
- XBRL concept names vary by company; the mapping handles common variations
- Values are typically in dollars (check the filing for currency/units)